# RoPE's One Property, and Where It Stops Holding

**Topic:** Rotary embeddings &nbsp;·&nbsp; **Difficulty:** Hard

Rotary embeddings rotate query and key by an angle proportional to position. The
consequence, the entire point, is that the dot product after rotation depends
only on the *difference* of the two positions, so the model gets relative
positions for free with no learned position table.

Verify the property numerically, then push past the training length and watch
what actually goes wrong when a model is asked to extrapolate.

## The data (created in the next cell)

- `d = 64` head dimension, `base = 10000`.
- `q`, `k`: fixed random vectors.
- The "training length" is 2048, and the extrapolation test uses position 4096.

## Your task

With **NumPy only**:

1. Implement `rope(x, pos)`: split `x` into pairs, and rotate pair `j` by
   `pos * base^(-2j/d)`, using the standard rotation of the pair
   `(x[2j], x[2j+1])`: the first component becomes
   `x[2j] * cos(theta) - x[2j+1] * sin(theta)` and the second becomes
   `x[2j] * sin(theta) + x[2j+1] * cos(theta)`, with
   `theta = pos * base^(-2j/d)`.
2. `rel_error`: for 200 random position pairs `(m, n)` inside the training
   length, compare `rope(q, m) . rope(k, n)` against `rope(q, m - n) . rope(k, 0)`.
   Report the largest absolute difference.
3. `wavelength_max`: the wavelength of the slowest-rotating frequency,
   `2 * pi * base^((d - 2) / d)`.
4. `frac_unseen`: the fraction of the slowest frequency's period that a
   2048-token training window covers. If this is well below 1, the model has
   never seen a full rotation of that dimension.
5. Set `answers` to a dict with `"rel_error"`, `"wavelength_max"`,
   `"frac_unseen"`, and `"dot_at_4096"`: `rope(q, 4096) . rope(k, 0)`, the
   dot product at relative distance 4096, which is outside the training range.

## Evaluation

Compared to the reference at a relative tolerance of 1e-6. The evaluation cell
also checks the relative-position property holds to better than 1e-9, with a
measured error above 1e-13 (rounding never leaves it at exactly zero), and that
frac_unseen is below 0.5, meaning the slowest frequency's period is less than
half covered by the training window.

In [ ]:
import numpy as np

rng = np.random.default_rng(4009)
d, base = 64, 10000.0
q = rng.normal(0, 1, d)
k = rng.normal(0, 1, d)
train_len = 2048
print(f"head dim {d}, base {base:g}, training length {train_len}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to implement `rope` and fill `answers`, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("rel_error", [], 3.306244167333716e-13, -5.535909598703742e-14, 1e-06),
    ("wavelength_max", [], 47117.24278016739, -7889.217594644021, 1e-06),
    ("frac_unseen", [], 0.04346604086226465, -0.007277867593823495, 1e-06),
    ("dot_at_4096", [], -10.594529388067595, 1.7739269686319088, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "rel_error" in answers:
    _plausible = 1e-13 < answers["rel_error"] < 1e-9
    _under = answers["frac_unseen"] < 0.5
    print(f"  {'relative property exact':<24} {'ok' if _plausible else 'wrong'}  "
          f"(max error {answers['rel_error']:.2e})")
    print(f"  {'slowest freq undertrained':<24} {'ok' if _under else 'wrong'}  "
          f"(window covers {answers['frac_unseen']:.1%} of its period)")
    _ok = _ok and _plausible and _under

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The property holds because a rotation by `m` followed by the inverse rotation by
`n` is a rotation by `m - n`, and the dot product is invariant to rotating both
vectors together. It is exact: the error is machine epsilon, not an
approximation. The extrapolation failure is not about the property breaking: it
is that the lowest-frequency dimensions have a wavelength far longer than the
training window, so the model has literally never observed those angles and has no
idea what they mean. That is precisely what position interpolation and NTK-aware
scaling address.

```python
import numpy as np

inv_freq = base ** (-np.arange(0, d, 2) / d)

def rope(x, pos):
    ang = pos * inv_freq
    c, s = np.cos(ang), np.sin(ang)
    xe, xo = x[0::2], x[1::2]
    out = np.empty_like(x)
    out[0::2] = xe * c - xo * s
    out[1::2] = xe * s + xo * c
    return out

rs = np.random.default_rng(7)
worst = 0.0
for _ in range(200):
    m = int(rs.integers(0, train_len))
    n = int(rs.integers(0, train_len))
    lhs = float(rope(q, m) @ rope(k, n))
    rhs = float(rope(q, m - n) @ rope(k, 0))
    worst = max(worst, abs(lhs - rhs))

wavelength_max = float(2 * np.pi * base ** ((d - 2) / d))

answers = {
    "rel_error": float(worst),
    "wavelength_max": wavelength_max,
    "frac_unseen": float(train_len / wavelength_max),
    "dot_at_4096": float(rope(q, 4096) @ rope(k, 0)),
}
```